In [2]:
from google.colab import files

print("Upload your extracted RAVDESS folder as a ZIP file")
uploaded = files.upload()

Upload your extracted RAVDESS folder as a ZIP file


Saving Audio_Song_Actors_01-24.zip to Audio_Song_Actors_01-24.zip


In [3]:
import zipfile
import os

zip_path = "Audio_Song_Actors_01-24.zip"
extract_path = "ravdess_data"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction complete.")
print("Top-level folders:", os.listdir(extract_path))

Extraction complete.
Top-level folders: ['Actor_06', 'Actor_08', 'Actor_24', 'Actor_22', 'Actor_10', 'Actor_04', 'Actor_11', 'Actor_14', 'Actor_09', 'Actor_18', 'Actor_17', 'Actor_19', 'Actor_21', 'Actor_02', 'Actor_12', 'Actor_23', 'Actor_20', 'Actor_07', 'Actor_05', 'Actor_13', 'Actor_16', 'Actor_03', 'Actor_15', 'Actor_01']


In [4]:
wav_count = 0
for root, _, files in os.walk(extract_path):
    for f in files:
        if f.lower().endswith(".wav"):
            wav_count += 1

print("Total WAV files found:", wav_count)

Total WAV files found: 1012


In [5]:
import os
import numpy as np
import pandas as pd
import librosa

from tqdm import tqdm

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

# Extracted folder (you already have this)
RAVDESS_ROOT = "ravdess_data"

# RAVDESS emotion codes
EMOTION_MAP = {
    1: "neutral",
    2: "calm",
    3: "happy",
    4: "sad",
    5: "angry",
    6: "fearful",
    7: "disgust",
    8: "surprised",
}

# Stress proxy mapping (you can tweak later)
STRESS_EMOTIONS = {"angry", "fearful", "sad"}       # label 1
NOSTRESS_EMOTIONS = {"neutral", "calm", "happy"}    # label 0

def parse_ravdess_filename(fname: str):
    """
    Format: 03-02-EMOTION-INTENSITY-STATEMENT-REPETITION-ACTOR.wav  (song=02 in 2nd field)
    We only need emotion code and actor id.
    """
    base = os.path.splitext(os.path.basename(fname))[0]
    parts = base.split("-")
    if len(parts) != 7:
        return None
    try:
        emotion_code = int(parts[2])
        actor_id = int(parts[6])
    except:
        return None
    emotion = EMOTION_MAP.get(emotion_code)
    if emotion is None:
        return None
    return emotion, actor_id

def extract_features(path: str, sr: int = 16000):
    """
    No major shortcuts. Uses MFCC mean/std + chroma + RMS + ZCR.
    """
    y, sr = librosa.load(path, sr=sr)
    y, _ = librosa.effects.trim(y, top_db=25)

    # Handle very short audio
    if len(y) < sr * 1.0:
        y = np.pad(y, (0, int(sr * 1.0) - len(y)))

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    mfcc_mean = mfcc.mean(axis=1)
    mfcc_std = mfcc.std(axis=1)

    chroma = librosa.feature.chroma_stft(y=y, sr=sr).mean(axis=1)

    zcr = float(librosa.feature.zero_crossing_rate(y).mean())
    rms = float(librosa.feature.rms(y=y).mean())

    feat = np.hstack([mfcc_mean, mfcc_std, chroma, [zcr, rms]]).astype(np.float32)
    return feat

In [6]:
wav_paths = []
for root, _, files in os.walk(RAVDESS_ROOT):
    for f in files:
        if f.lower().endswith(".wav"):
            wav_paths.append(os.path.join(root, f))

print("Total wav files found:", len(wav_paths))

rows = []
for p in wav_paths:
    parsed = parse_ravdess_filename(os.path.basename(p))
    if not parsed:
        continue
    emotion, actor = parsed

    if emotion in STRESS_EMOTIONS:
        label = 1
    elif emotion in NOSTRESS_EMOTIONS:
        label = 0
    else:
        continue  # drop disgust/surprised by default

    rows.append({"path": p, "emotion": emotion, "label": label, "actor": actor})

df = pd.DataFrame(rows)
print("Usable samples:", df.shape)
print("\nLabel distribution:\n", df["label"].value_counts())
print("\nEmotion distribution:\n", df["emotion"].value_counts())
df.head()

Total wav files found: 1012
Usable samples: (1012, 4)

Label distribution:
 label
1    552
0    460
Name: count, dtype: int64

Emotion distribution:
 emotion
happy      184
sad        184
fearful    184
angry      184
calm       184
neutral     92
Name: count, dtype: int64


,path,emotion,label,actor
0,ravdess_data/Actor_06/03-02-03-01-02-02-06.wav,happy,0,6
1,ravdess_data/Actor_06/03-02-04-01-01-02-06.wav,sad,1,6
2,ravdess_data/Actor_06/03-02-06-02-02-02-06.wav,fearful,1,6
3,ravdess_data/Actor_06/03-02-05-02-02-02-06.wav,angry,1,6
4,ravdess_data/Actor_06/03-02-04-02-02-01-06.wav,sad,1,6


In [7]:
X_list, y_list, g_list = [], [], []

for _, r in tqdm(df.iterrows(), total=len(df)):
    X_list.append(extract_features(r["path"]))
    y_list.append(int(r["label"]))
    g_list.append(int(r["actor"]))

X = np.vstack(X_list)
y = np.array(y_list, dtype=int)
groups = np.array(g_list, dtype=int)

print("X shape:", X.shape, "y shape:", y.shape)

100%|██████████| 1012/1012 [00:39<00:00, 25.59it/s]

X shape: (1012, 94) y shape: (1012,)


In [8]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

print("Train size:", X_train.shape, "Test size:", X_test.shape)
print("Train labels:", np.bincount(y_train), "Test labels:", np.bincount(y_test))

model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", C=3.0, gamma="scale", probability=True, class_weight="balanced", random_state=42))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, digits=4))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob), 4))

Train size: (792, 94) Test size: (220, 94)
Train labels: [360 432] Test labels: [100 120]

Confusion Matrix:
 [[ 92   8]
 [  0 120]]

Classification Report:
               precision    recall  f1-score   support

           0     1.0000    0.9200    0.9583       100
           1     0.9375    1.0000    0.9677       120

    accuracy                         0.9636       220
   macro avg     0.9688    0.9600    0.9630       220
weighted avg     0.9659    0.9636    0.9635       220

ROC-AUC: 1.0


In [9]:
def stress_level(prob):
    if prob < 0.40:
        return "Low"
    elif prob < 0.70:
        return "Moderate"
    else:
        return "High"

def predict_stress_from_audio(audio_path: str):
    feat = extract_features(audio_path)
    prob = float(model.predict_proba(feat.reshape(1, -1))[0, 1])  # P(stress=1)
    pred = int(prob >= 0.5)
    return {
        "file": os.path.basename(audio_path),
        "predicted_label": pred,
        "stress_probability": round(prob, 4),
        "stress_level": stress_level(prob)
    }

# Example: test first file in df
print(predict_stress_from_audio(df.iloc[0]["path"]))

{'file': '03-02-03-01-02-02-06.wav', 'predicted_label': 0, 'stress_probability': 0.0002, 'stress_level': 'Low'}


In [10]:
import random
from pprint import pprint

# Pick a random file from the dataframe you already built
row = df.sample(1, random_state=random.randint(0, 10_000)).iloc[0]
audio_path = row["path"]

pprint({
    "chosen_file": audio_path,
    "true_emotion": row["emotion"],
    "true_proxy_label": row["label"]  # 1=stress proxy, 0=no-stress proxy
})

pprint(predict_stress_from_audio(audio_path))

{'chosen_file': 'ravdess_data/Actor_09/03-02-06-01-02-01-09.wav',
 'true_emotion': 'fearful',
 'true_proxy_label': np.int64(1)}
{'file': '03-02-06-01-02-01-09.wav',
 'predicted_label': 1,
 'stress_level': 'High',
 'stress_probability': 0.9855}


In [11]:
for i, p in enumerate(df["path"].head(20)):
    print(i, p)

0 ravdess_data/Actor_06/03-02-03-01-02-02-06.wav
1 ravdess_data/Actor_06/03-02-04-01-01-02-06.wav
2 ravdess_data/Actor_06/03-02-06-02-02-02-06.wav
3 ravdess_data/Actor_06/03-02-05-02-02-02-06.wav
4 ravdess_data/Actor_06/03-02-04-02-02-01-06.wav
5 ravdess_data/Actor_06/03-02-01-01-02-01-06.wav
6 ravdess_data/Actor_06/03-02-03-01-01-01-06.wav
7 ravdess_data/Actor_06/03-02-02-01-02-02-06.wav
8 ravdess_data/Actor_06/03-02-02-01-02-01-06.wav
9 ravdess_data/Actor_06/03-02-02-02-01-02-06.wav
10 ravdess_data/Actor_06/03-02-03-02-02-01-06.wav
11 ravdess_data/Actor_06/03-02-06-02-01-01-06.wav
12 ravdess_data/Actor_06/03-02-06-01-01-01-06.wav
13 ravdess_data/Actor_06/03-02-03-02-02-02-06.wav
14 ravdess_data/Actor_06/03-02-03-02-01-01-06.wav
15 ravdess_data/Actor_06/03-02-01-01-02-02-06.wav
16 ravdess_data/Actor_06/03-02-04-01-02-01-06.wav
17 ravdess_data/Actor_06/03-02-06-02-02-01-06.wav
18 ravdess_data/Actor_06/03-02-05-01-01-02-06.wav
19 ravdess_data/Actor_06/03-02-01-01-01-01-06.wav


In [13]:
# Show first 20 files
for i, row in df.head(20).iterrows():
    print(i, row["emotion"], row["path"])

0 happy ravdess_data/Actor_06/03-02-03-01-02-02-06.wav
1 sad ravdess_data/Actor_06/03-02-04-01-01-02-06.wav
2 fearful ravdess_data/Actor_06/03-02-06-02-02-02-06.wav
3 angry ravdess_data/Actor_06/03-02-05-02-02-02-06.wav
4 sad ravdess_data/Actor_06/03-02-04-02-02-01-06.wav
5 neutral ravdess_data/Actor_06/03-02-01-01-02-01-06.wav
6 happy ravdess_data/Actor_06/03-02-03-01-01-01-06.wav
7 calm ravdess_data/Actor_06/03-02-02-01-02-02-06.wav
8 calm ravdess_data/Actor_06/03-02-02-01-02-01-06.wav
9 calm ravdess_data/Actor_06/03-02-02-02-01-02-06.wav
10 happy ravdess_data/Actor_06/03-02-03-02-02-01-06.wav
11 fearful ravdess_data/Actor_06/03-02-06-02-01-01-06.wav
12 fearful ravdess_data/Actor_06/03-02-06-01-01-01-06.wav
13 happy ravdess_data/Actor_06/03-02-03-02-02-02-06.wav
14 happy ravdess_data/Actor_06/03-02-03-02-01-01-06.wav
15 neutral ravdess_data/Actor_06/03-02-01-01-02-02-06.wav
16 sad ravdess_data/Actor_06/03-02-04-01-02-01-06.wav
17 fearful ravdess_data/Actor_06/03-02-06-02-02-01-06.wav

In [15]:
from pprint import pprint

row = df.loc[9]
pprint({
    "emotion": row["emotion"],
    "proxy_label": row["label"]
})

pprint(predict_stress_from_audio(row["path"]))

{'emotion': 'calm', 'proxy_label': np.int64(0)}
{'file': '03-02-02-02-01-02-06.wav',
 'predicted_label': 0,
 'stress_level': 'Low',
 'stress_probability': 0.0007}
